In [1]:
import torch
import torch.nn as nn


In [2]:

torch.manual_seed(0)
n, d_model, d_k, d_v = 3, 8, 4, 4
X = torch.randn(n, d_model)                       # (3, 8)

W_Q = nn.Linear(d_model, d_k, bias=False)         # ağırlık (4, 8) saklanır, X @ W^T uygular
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_v, bias=False)

Q, K, V = W_Q(X), W_K(X), W_V(X)
print("X:", X.shape, "Q:", Q.shape, "K:", K.shape, "V:", V.shape)

X: torch.Size([3, 8]) Q: torch.Size([3, 4]) K: torch.Size([3, 4]) V: torch.Size([3, 4])


In [3]:
import torch

d_k = 4
scores = torch.tensor([2.0, 0.5, 1.0])
print("ölçeksiz softmax:", torch.softmax(scores, -1).round(decimals=3))
print("ölçekli softmax :", torch.softmax(scores / d_k**0.5, -1).round(decimals=3))

# Neden gerekli: d_k büyüyünce dot product varyansı büyür
torch.manual_seed(0)
for d in [4, 64, 1024]:
    q, k = torch.randn(10000, d), torch.randn(10000, d)
    dots = (q * k).sum(-1)
    print(f"d_k={d:5d}  std(dot)={dots.std().item():.2f}  std(dot/sqrt(d))={(dots/d**0.5).std().item():.2f}")

ölçeksiz softmax: tensor([0.6290, 0.1400, 0.2310])
ölçekli softmax : tensor([0.4810, 0.2270, 0.2920])
d_k=    4  std(dot)=1.99  std(dot/sqrt(d))=1.00
d_k=   64  std(dot)=8.01  std(dot/sqrt(d))=1.00
d_k= 1024  std(dot)=32.11  std(dot/sqrt(d))=1.00


In [4]:
import numpy as np

def softmax(x):
    # Sayısal kararlılık için maksimumu çıkar
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    # Normalize et
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = np.dot(Q, K.T)                 # Adım 1: Q (n x d_k) @ K^T (d_k x n)
    scaled_scores = scores / np.sqrt(d_k)   # Adım 2: ölçekleme
    if mask is not None:
        scaled_scores = scaled_scores + mask  # Adım 3: maske (varsa)
    attention_weights = softmax(scaled_scores)   # Adım 4: softmax
    output = np.dot(attention_weights, V)        # Adım 5: value karıştırma
    return output, attention_weights

K = np.array([[1.0, 1.0, 1.0, 1.0],      # animal
              [0.5, 0.5, 0.0, 0.0],      # street
              [1.0, 0.0, 0.5, 0.5]])     # tired
V = np.array([[1, 0, 0, 1],              # animal
              [0, 1, 0, 1],              # street
              [1, 1, 1, 0]])             # tired
Q = np.array([[0.5, 0.5, 0.5, 0.5]])     # "it"

output, weights = scaled_dot_product_attention(Q, K, V)
print("Attention ağırlıkları:", weights)
print("Çıktı vektörü:", output)

Attention ağırlıkları: [[0.48102426 0.22721977 0.29175596]]
Çıktı vektörü: [[0.77278023 0.51897574 0.29175596 0.70824404]]


In [5]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (n_q, d_k)   K: (n_k, d_k)   V: (n_k, d_v)
    mask: (n_q, n_k) toplamsal maske (0 veya -inf)
    döner: output (n_q, d_v), weights (n_q, n_k)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1)          # (n_q, d_k)@(d_k, n_k) -> (n_q, n_k)
    scores = scores / d_k ** 0.5              # ölçekleme
    if mask is not None:
        scores = scores + mask                # -inf olan yerler engellenir
    weights = F.softmax(scores, dim=-1)       # satır bazlı: her satır toplamı 1
    output = weights @ V                      # (n_q, n_k)@(n_k, d_v) -> (n_q, d_v)
    return output, weights

K = torch.tensor([[1.0, 1.0, 1.0, 1.0],
                  [0.5, 0.5, 0.0, 0.0],
                  [1.0, 0.0, 0.5, 0.5]])      # (3, 4)
V = torch.tensor([[1., 0., 0., 1.],
                  [0., 1., 0., 1.],
                  [1., 1., 1., 0.]])          # (3, 4)
Q = torch.tensor([[0.5, 0.5, 0.5, 0.5]])      # (1, 4)

out, w = scaled_dot_product_attention(Q, K, V)
print("skor (ham):", (Q @ K.T))               # (1,3)
print("ağırlıklar:", w.round(decimals=3), "| shape:", w.shape)
print("çıktı      :", out.round(decimals=3), "| shape:", out.shape)
print("ağırlık toplamı:", w.sum().item())

skor (ham): tensor([[2.0000, 0.5000, 1.0000]])
ağırlıklar: tensor([[0.4810, 0.2270, 0.2920]]) | shape: torch.Size([1, 3])
çıktı      : tensor([[0.7730, 0.5190, 0.2920, 0.7080]]) | shape: torch.Size([1, 4])
ağırlık toplamı: 1.0


In [6]:
Q_all = torch.tensor([[0.5, 0.5, 0.5, 0.5],
                      [1.0, 0.0, 0.0, 0.0],
                      [0.0, 0.0, 1.0, 1.0]])    # (3, 4)  uydurma 3 query
out_all, w_all = scaled_dot_product_attention(Q_all, K, V)
print(w_all.shape, out_all.shape)               # (3,3) ve (3,4)
print(w_all.sum(-1))                            # her satır toplamı 1

torch.Size([3, 3]) torch.Size([3, 4])
tensor([1., 1., 1.])
